# SupportOps AI - RAG Retrieval Pipeline

## Objective

Build the retrieval layer for the SupportOps AI knowledge assistant.

### Pipeline

Support Policies
→ Document Loading
→ Chunking
→ Embeddings
→ Vector Database
→ Semantic Search
→ Relevant Policy Sections

The initial implementation uses Sentence Transformers and ChromaDB.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb

print("RAG libraries imported successfully!")

RAG libraries imported successfully!


In [2]:
KNOWLEDGE_BASE_DIR = Path(
    "../knowledge_base"
)

documents = []

for file_path in sorted(
    KNOWLEDGE_BASE_DIR.glob("*.md")
):
    
    text = file_path.read_text(
        encoding="utf-8"
    )

    documents.append({
        "source": file_path.name,
        "content": text
    })

print(
    "Documents loaded:",
    len(documents)
)

Documents loaded: 8


In [3]:
for document in documents:
    print("=" * 70)
    print("SOURCE:", document["source"])
    print(document["content"][:500])

SOURCE: 01_card_payments.md
# NovaBank Card Payment Policy

## 1. Card Payment Declined

If a customer reports that a card payment was declined, support agents should
first verify that the card is active and has not been frozen.

Agents should ask the customer to confirm that sufficient funds are available
and that the merchant accepts the card type.

If repeated card payments are declined despite sufficient funds and an active
card, the issue should be escalated to the Card Payments team.

## 2. Duplicate Card Payment

A 
SOURCE: 02_refunds.md
# NovaBank Refund Policy

## 1. Requesting a Refund

NovaBank does not normally initiate merchant refunds directly.

Customers should first contact the merchant and request a refund.

If the merchant confirms that a refund has been issued, the customer should
retain any confirmation email or receipt provided by the merchant.

## 2. Refund Not Showing

Merchant refunds may take several business days to appear in the customer's
account after they 

Create a simple Markdown chunker

In [4]:
def chunk_markdown_document(
    text,
    source
):
    
    chunks = []

    sections = text.split("\n## ")

    for index, section in enumerate(sections):

        section = section.strip()

        if not section:
            continue

        if index == 0:
            chunk_text = section
        else:
            chunk_text = "## " + section

        chunks.append({
            "source": source,
            "chunk_id": f"{source}_{index}",
            "text": chunk_text
        })

    return chunks

In [5]:
all_chunks = []

for document in documents:

    document_chunks = (
        chunk_markdown_document(
            document["content"],
            document["source"]
        )
    )

    all_chunks.extend(
        document_chunks
    )

print(
    "Total chunks:",
    len(all_chunks)
)

Total chunks: 15


In [6]:
chunks_df = pd.DataFrame(
    all_chunks
)

chunks_df[
    ["source", "chunk_id", "text"]
]

,source,chunk_id,text
0,01_card_payments.md,01_card_payments.md_0,# NovaBank Card Payment Policy
1,01_card_payments.md,01_card_payments.md_1,## 1. Card Payment Declined\n\nIf a customer r...
2,01_card_payments.md,01_card_payments.md_2,## 2. Duplicate Card Payment\n\nA duplicate ca...
3,01_card_payments.md,01_card_payments.md_3,## 3. Card Payment Not Recognised\n\nIf a cust...
4,01_card_payments.md,01_card_payments.md_4,## 4. Pending Card Payment\n\nPending card pay...
5,02_refunds.md,02_refunds.md_0,# NovaBank Refund Policy
6,02_refunds.md,02_refunds.md_1,## 1. Requesting a Refund\n\nNovaBank does not...
7,02_refunds.md,02_refunds.md_2,## 2. Refund Not Showing\n\nMerchant refunds m...
8,02_refunds.md,02_refunds.md_3,## 3. Duplicate Payment Refund\n\nIf a custome...
9,02_refunds.md,02_refunds.md_4,## 4. Refund Escalation\n\nRefund cases should...


Load the embedding model

In [7]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print(
    "Embedding model loaded successfully!"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


Generate one embedding

In [8]:
sample_sentence = (
    "I was charged twice for the same card payment."
)

sample_embedding = embedding_model.encode(
    sample_sentence
)

print(
    "Embedding shape:",
    sample_embedding.shape
)

print(
    sample_embedding[:10]
)

Embedding shape: (384,)
[ 0.01639304  0.0196584   0.02213017  0.03730961  0.02157849 -0.08886568
  0.01123362 -0.05081679  0.05950845 -0.04067073]


Compare semantic similarity

In [9]:
sentences = [
    "I was charged twice for the same card payment.",
    "There is a duplicate transaction on my card.",
    "I forgot my PIN.",
    "My transfer is still pending."
]

embeddings = embedding_model.encode(
    sentences,
    normalize_embeddings=True
)

similarity_matrix = (
    embeddings @ embeddings.T
)

pd.DataFrame(
    similarity_matrix,
    index=sentences,
    columns=sentences
)

,I was charged twice for the same card payment.,There is a duplicate transaction on my card.,I forgot my PIN.,My transfer is still pending.
I was charged twice for the same card payment.,1.000000,0.692613,0.273494,0.324774
There is a duplicate transaction on my card.,0.692613,1.000000,0.300990,0.373506
I forgot my PIN.,0.273494,0.300990,1.000000,0.154796
My transfer is still pending.,0.324774,0.373506,0.154796,1.000000


Create a persistent Chroma client

In [10]:
CHROMA_PATH = "../rag/chroma_db"

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

print("ChromaDB client created!")

ChromaDB client created!


Create a collection

In [11]:
COLLECTION_NAME = "novabank_support_policies"

try:
    chroma_client.delete_collection(
        name=COLLECTION_NAME
    )
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={
        "hnsw:space": "cosine"
    }
)

print(
    "Collection created:",
    COLLECTION_NAME
)

Collection created: novabank_support_policies


Generate embeddings for all chunks

In [12]:
chunk_texts = [
    chunk["text"]
    for chunk in all_chunks
]

In [13]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True
)

print(
    "Chunk embeddings shape:",
    chunk_embeddings.shape
)

Chunk embeddings shape: (15, 384)


Prepare metadata

In [14]:
chunk_ids = [
    chunk["chunk_id"]
    for chunk in all_chunks
]

chunk_metadata = [
    {
        "source": chunk["source"],
        "chunk_id": chunk["chunk_id"]
    }
    for chunk in all_chunks
]

Add everything to ChromaDB

In [15]:
collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    embeddings=chunk_embeddings.tolist(),
    metadatas=chunk_metadata
)

print(
    "Chunks stored in ChromaDB:",
    collection.count()
)

Chunks stored in ChromaDB: 15


Run your first semantic search

In [16]:
query = (
    "I was charged twice for the same card purchase."
)

In [17]:
#Generate its embedding
query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

In [18]:
#then query chroma
results = collection.query(
    query_embeddings=[
        query_embedding.tolist()
    ],
    n_results=3
)

results

{'ids': [['02_refunds.md_3',
   '01_card_payments.md_2',
   '01_card_payments.md_3']],
 'embeddings': None,
 'documents': [['## 3. Duplicate Payment Refund\n\nIf a customer was charged twice for the same completed card transaction, the\ncase should follow the duplicate-payment investigation process.\n\nA duplicate-payment investigation should include the merchant name,\ntransaction date, transaction amount, and evidence that both transactions\nrepresent the same purchase.',
   "## 2. Duplicate Card Payment\n\nA duplicate card payment occurs when the same merchant transaction appears more\nthan once on the customer's account.\n\nCustomers should first verify whether both transactions have fully completed.\nOne transaction may sometimes remain pending temporarily before disappearing.\n\nIf both transactions are completed and represent the same purchase, support\nagents should create a duplicate-payment investigation.\n\nThe customer should provide the merchant name, transaction date, tra

Make the results readable

In [19]:
for i in range(
    len(results["documents"][0])
):
    
    print("=" * 80)

    print(
        "RANK:",
        i + 1
    )

    print(
        "SOURCE:",
        results["metadatas"][0][i]["source"]
    )

    print(
        "DISTANCE:",
        round(
            results["distances"][0][i],
            4
        )
    )

    print("\nCONTENT:")
    print(
        results["documents"][0][i]
    )

    print()

RANK: 1
SOURCE: 02_refunds.md
DISTANCE: 0.4033

CONTENT:
## 3. Duplicate Payment Refund

If a customer was charged twice for the same completed card transaction, the
case should follow the duplicate-payment investigation process.

A duplicate-payment investigation should include the merchant name,
transaction date, transaction amount, and evidence that both transactions
represent the same purchase.

RANK: 2
SOURCE: 01_card_payments.md
DISTANCE: 0.4586

CONTENT:
## 2. Duplicate Card Payment

A duplicate card payment occurs when the same merchant transaction appears more
than once on the customer's account.

Customers should first verify whether both transactions have fully completed.
One transaction may sometimes remain pending temporarily before disappearing.

If both transactions are completed and represent the same purchase, support
agents should create a duplicate-payment investigation.

The customer should provide the merchant name, transaction date, transaction
amount, and the las

Create a clean retrieval function

In [20]:
def retrieve_context(
    query,
    top_k=3
):
    
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    retrieved_chunks = []

    for i in range(
        len(results["documents"][0])
    ):
        
        distance = (
            results["distances"][0][i]
        )

        retrieved_chunks.append({
            "rank": i + 1,

            "source":
                results["metadatas"][0][i][
                    "source"
                ],

            "chunk_id":
                results["metadatas"][0][i][
                    "chunk_id"
                ],

            "text":
                results["documents"][0][i],

            "distance":
                round(float(distance), 4),

            "similarity":
                round(
                    1 - float(distance),
                    4
                )
        })

    return retrieved_chunks

In [21]:
retrieve_context(
    "I was charged twice for one card transaction."
)

[{'rank': 1,
  'source': '02_refunds.md',
  'chunk_id': '02_refunds.md_3',
  'text': '## 3. Duplicate Payment Refund\n\nIf a customer was charged twice for the same completed card transaction, the\ncase should follow the duplicate-payment investigation process.\n\nA duplicate-payment investigation should include the merchant name,\ntransaction date, transaction amount, and evidence that both transactions\nrepresent the same purchase.',
  'distance': 0.4191,
  'similarity': 0.5809},
 {'rank': 2,
  'source': '01_card_payments.md',
  'chunk_id': '01_card_payments.md_2',
  'text': "## 2. Duplicate Card Payment\n\nA duplicate card payment occurs when the same merchant transaction appears more\nthan once on the customer's account.\n\nCustomers should first verify whether both transactions have fully completed.\nOne transaction may sometimes remain pending temporarily before disappearing.\n\nIf both transactions are completed and represent the same purchase, support\nagents should create a du

Turn retrieval results into a DataFrame

In [22]:
retrieval_results = retrieve_context(
    "I was charged twice for one card transaction."
)

pd.DataFrame(
    retrieval_results
)[
    [
        "rank",
        "source",
        "chunk_id",
        "similarity"
    ]
]

,rank,source,chunk_id,similarity
0,1,02_refunds.md,02_refunds.md_3,0.5809
1,2,01_card_payments.md,01_card_payments.md_2,0.5391
2,3,01_card_payments.md,01_card_payments.md_3,0.4270


Test different business questions

In [24]:
#duplicate card payment
retrieve_context(
    "I was charged twice for the same transaction."
)

[{'rank': 1,
  'source': '02_refunds.md',
  'chunk_id': '02_refunds.md_3',
  'text': '## 3. Duplicate Payment Refund\n\nIf a customer was charged twice for the same completed card transaction, the\ncase should follow the duplicate-payment investigation process.\n\nA duplicate-payment investigation should include the merchant name,\ntransaction date, transaction amount, and evidence that both transactions\nrepresent the same purchase.',
  'distance': 0.4589,
  'similarity': 0.5411},
 {'rank': 2,
  'source': '01_card_payments.md',
  'chunk_id': '01_card_payments.md_2',
  'text': "## 2. Duplicate Card Payment\n\nA duplicate card payment occurs when the same merchant transaction appears more\nthan once on the customer's account.\n\nCustomers should first verify whether both transactions have fully completed.\nOne transaction may sometimes remain pending temporarily before disappearing.\n\nIf both transactions are completed and represent the same purchase, support\nagents should create a du

In [25]:
#missing refund
retrieve_context(
    "The merchant refunded me but the money still has not appeared."
)

[{'rank': 1,
  'source': '02_refunds.md',
  'chunk_id': '02_refunds.md_2',
  'text': "## 2. Refund Not Showing\n\nMerchant refunds may take several business days to appear in the customer's\naccount after they are processed.\n\nCustomers should normally allow up to ten business days for an issued refund to\nappear.\n\nIf the refund has not appeared after ten business days, the customer should\nprovide proof that the merchant processed the refund.\n\nSupport agents should then escalate the case to the Card Payments team for\ninvestigation.",
  'distance': 0.4048,
  'similarity': 0.5952},
 {'rank': 2,
  'source': '02_refunds.md',
  'chunk_id': '02_refunds.md_1',
  'text': '## 1. Requesting a Refund\n\nNovaBank does not normally initiate merchant refunds directly.\n\nCustomers should first contact the merchant and request a refund.\n\nIf the merchant confirms that a refund has been issued, the customer should\nretain any confirmation email or receipt provided by the merchant.',
  'distanc

In [26]:
#pending transfer
retrieve_context(
    "My bank transfer has been pending for three days."
)

[{'rank': 1,
  'source': '03_bank_transfers.md',
  'chunk_id': '03_bank_transfers.md_1',
  'text': '## 1. Pending Transfer\n\nA bank transfer may remain pending while payment checks are completed.\n\nCustomers should allow up to two business days for a standard pending transfer\nto complete.\n\nIf the transfer remains pending for more than two business days, support agents\nshould escalate the case to the Transfers team.',
  'distance': 0.2537,
  'similarity': 0.7463},
 {'rank': 2,
  'source': '03_bank_transfers.md',
  'chunk_id': '03_bank_transfers.md_4',
  'text': '## 4. Cancelling a Transfer\n\nCompleted bank transfers cannot normally be cancelled automatically.\n\nIf a transfer is still pending, support agents may check whether cancellation\nis available.\n\nIf the transfer has already completed, the customer should contact the recipient\nand the Transfers team should be consulted where appropriate.',
  'distance': 0.4967,
  'similarity': 0.5033},
 {'rank': 3,
  'source': '01_card_

In [27]:
#unknown card payment
retrieve_context(
    "There is a card transaction on my account that I do not recognise."
)

[{'rank': 1,
  'source': '01_card_payments.md',
  'chunk_id': '01_card_payments.md_3',
  'text': '## 3. Card Payment Not Recognised\n\nIf a customer does not recognise a card payment, the card should be temporarily\nfrozen while the transaction is investigated.\n\nThe customer should confirm whether another authorised cardholder may have made\nthe payment.\n\nIf the transaction remains unrecognised, the case should be escalated to the\nFraud and Security team.',
  'distance': 0.3665,
  'similarity': 0.6335},
 {'rank': 2,
  'source': '01_card_payments.md',
  'chunk_id': '01_card_payments.md_2',
  'text': "## 2. Duplicate Card Payment\n\nA duplicate card payment occurs when the same merchant transaction appears more\nthan once on the customer's account.\n\nCustomers should first verify whether both transactions have fully completed.\nOne transaction may sometimes remain pending temporarily before disappearing.\n\nIf both transactions are completed and represent the same purchase, support

Create a retrieval test table

In [28]:
retrieval_tests = [
    {
        "question":
            "I was charged twice for the same transaction.",
        "expected_source":
            "01_card_payments.md"
    },
    {
        "question":
            "The merchant refunded me but the money has not arrived.",
        "expected_source":
            "02_refunds.md"
    },
    {
        "question":
            "My bank transfer has been pending for three days.",
        "expected_source":
            "03_bank_transfers.md"
    },
    {
        "question":
            "I don't recognise a payment made with my card.",
        "expected_source":
            "01_card_payments.md"
    }
]

In [29]:
evaluation_rows = []

for test in retrieval_tests:

    retrieved = retrieve_context(
        test["question"],
        top_k=3
    )

    retrieved_sources = [
        item["source"]
        for item in retrieved
    ]

    hit = (
        test["expected_source"]
        in retrieved_sources
    )

    evaluation_rows.append({
        "question":
            test["question"],

        "expected_source":
            test["expected_source"],

        "top_1_source":
            retrieved_sources[0],

        "retrieved_sources":
            retrieved_sources,

        "hit_at_3":
            hit
    })

retrieval_evaluation_df = pd.DataFrame(
    evaluation_rows
)

retrieval_evaluation_df

,question,expected_source,top_1_source,retrieved_sources,hit_at_3
0,I was charged twice for the same transaction.,01_card_payments.md,02_refunds.md,"[02_refunds.md, 01_card_payments.md, 02_refund...",True
1,The merchant refunded me but the money has not...,02_refunds.md,02_refunds.md,"[02_refunds.md, 02_refunds.md, 02_refunds.md]",True
2,My bank transfer has been pending for three days.,03_bank_transfers.md,03_bank_transfers.md,"[03_bank_transfers.md, 03_bank_transfers.md, 0...",True
3,I don't recognise a payment made with my card.,01_card_payments.md,01_card_payments.md,"[01_card_payments.md, 01_card_payments.md, 01_...",True


Calculate Hit@3

In [30]:
hit_at_3 = (
    retrieval_evaluation_df[
        "hit_at_3"
    ].mean()
)

print(
    f"Retrieval Hit@3: "
    f"{hit_at_3:.2%}"
)

Retrieval Hit@3: 100.00%


Create a context builder

In [31]:
def build_rag_context(
    query,
    top_k=3
):
    
    retrieved = retrieve_context(
        query,
        top_k=top_k
    )

    context_blocks = []

    for item in retrieved:

        block = (
            f"[Source: {item['source']}]\n"
            f"{item['text']}"
        )

        context_blocks.append(block)

    context = "\n\n---\n\n".join(
        context_blocks
    )

    return context, retrieved

In [32]:
context, sources = build_rag_context(
    "What should we do when a customer is charged twice?"
)

print(context)

[Source: 02_refunds.md]
## 3. Duplicate Payment Refund

If a customer was charged twice for the same completed card transaction, the
case should follow the duplicate-payment investigation process.

A duplicate-payment investigation should include the merchant name,
transaction date, transaction amount, and evidence that both transactions
represent the same purchase.

---

[Source: 01_card_payments.md]
## 2. Duplicate Card Payment

A duplicate card payment occurs when the same merchant transaction appears more
than once on the customer's account.

Customers should first verify whether both transactions have fully completed.
One transaction may sometimes remain pending temporarily before disappearing.

If both transactions are completed and represent the same purchase, support
agents should create a duplicate-payment investigation.

The customer should provide the merchant name, transaction date, transaction
amount, and the last four digits of the affected card.

Duplicate-payment invest

Build the full RAG prompt

In [33]:
def build_rag_prompt(
    question,
    context
):
    
    prompt = f"""
You are SupportOps AI, an internal customer-support assistant for NovaBank.

Your job is to help support agents resolve customer issues using ONLY the
information provided in the retrieved NovaBank policy context.

Rules:
1. Do not invent policies, timelines, fees, procedures, or requirements.
2. Base your answer only on the retrieved context.
3. If the context does not contain enough information, clearly say:
   "The available policy context does not provide enough information."
4. Give clear, actionable steps for the support agent.
5. Mention relevant timelines when they are explicitly stated in the context.
6. Cite the source document used for each important recommendation.
7. Do not claim that an action has already been completed.
8. Keep the response concise and professional.

RETRIEVED POLICY CONTEXT
------------------------
{context}

CUSTOMER QUESTION
-----------------
{question}

SUPPORT RECOMMENDATION
----------------------
"""
    
    return prompt.strip()

In [34]:
question = (
    "I was charged twice for the same card transaction. "
    "What should I do?"
)

context, retrieved_sources = build_rag_context(
    question,
    top_k=3
)

prompt = build_rag_prompt(
    question,
    context
)

print(prompt)

You are SupportOps AI, an internal customer-support assistant for NovaBank.

Your job is to help support agents resolve customer issues using ONLY the
information provided in the retrieved NovaBank policy context.

Rules:
1. Do not invent policies, timelines, fees, procedures, or requirements.
2. Base your answer only on the retrieved context.
3. If the context does not contain enough information, clearly say:
   "The available policy context does not provide enough information."
4. Give clear, actionable steps for the support agent.
5. Mention relevant timelines when they are explicitly stated in the context.
6. Cite the source document used for each important recommendation.
7. Do not claim that an action has already been completed.
8. Keep the response concise and professional.

RETRIEVED POLICY CONTEXT
------------------------
[Source: 02_refunds.md]
## 3. Duplicate Payment Refund

If a customer was charged twice for the same completed card transaction, the
case should follow the d

Add source labels more cleanly

In [35]:
def build_rag_context(
    query,
    top_k=3
):
    
    retrieved = retrieve_context(
        query,
        top_k=top_k
    )

    context_blocks = []

    for index, item in enumerate(
        retrieved,
        start=1
    ):
        
        block = (
            f"[Source {index}: "
            f"{item['source']} | "
            f"{item['chunk_id']}]\n"
            f"{item['text']}"
        )

        context_blocks.append(block)

    context = "\n\n---\n\n".join(
        context_blocks
    )

    return context, retrieved

Add source labels more cleanly

In [36]:
def build_rag_context(
    query,
    top_k=3
):
    
    retrieved = retrieve_context(
        query,
        top_k=top_k
    )

    context_blocks = []

    for index, item in enumerate(
        retrieved,
        start=1
    ):
        
        block = (
            f"[Source {index}: "
            f"{item['source']} | "
            f"{item['chunk_id']}]\n"
            f"{item['text']}"
        )

        context_blocks.append(block)

    context = "\n\n---\n\n".join(
        context_blocks
    )

    return context, retrieved

In [37]:
context, retrieved = build_rag_context(
    "I was charged twice for the same transaction."
)

print(context)

[Source 1: 02_refunds.md | 02_refunds.md_3]
## 3. Duplicate Payment Refund

If a customer was charged twice for the same completed card transaction, the
case should follow the duplicate-payment investigation process.

A duplicate-payment investigation should include the merchant name,
transaction date, transaction amount, and evidence that both transactions
represent the same purchase.

---

[Source 2: 01_card_payments.md | 01_card_payments.md_2]
## 2. Duplicate Card Payment

A duplicate card payment occurs when the same merchant transaction appears more
than once on the customer's account.

Customers should first verify whether both transactions have fully completed.
One transaction may sometimes remain pending temporarily before disappearing.

If both transactions are completed and represent the same purchase, support
agents should create a duplicate-payment investigation.

The customer should provide the merchant name, transaction date, transaction
amount, and the last four digits o

Add a retrieval confidence guard

In [38]:
def retrieve_context(
    query,
    top_k=3,
    min_similarity=None
):
    
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    retrieved_chunks = []

    for i in range(
        len(results["documents"][0])
    ):
        
        distance = float(
            results["distances"][0][i]
        )

        similarity = 1 - distance

        if (
            min_similarity is not None
            and similarity < min_similarity
        ):
            continue

        retrieved_chunks.append({
            "rank": i + 1,
            "source":
                results["metadatas"][0][i]["source"],
            "chunk_id":
                results["metadatas"][0][i]["chunk_id"],
            "text":
                results["documents"][0][i],
            "distance":
                round(distance, 4),
            "similarity":
                round(similarity, 4)
        })

    return retrieved_chunks

Test an out-of-domain question

In [39]:
out_of_domain_results = retrieve_context(
    "What is the weather in London tomorrow?",
    top_k=3
)

pd.DataFrame(
    out_of_domain_results
)[
    [
        "rank",
        "source",
        "chunk_id",
        "similarity"
    ]
]

,rank,source,chunk_id,similarity
0,1,03_bank_transfers.md,03_bank_transfers.md_0,0.0851
1,2,01_card_payments.md,01_card_payments.md_0,0.0716
2,3,03_bank_transfers.md,03_bank_transfers.md_1,0.0413


Test an out-of-domain question

In [40]:
out_of_domain_results = retrieve_context(
    "What is the weather in London tomorrow?",
    top_k=3
)

pd.DataFrame(
    out_of_domain_results
)[
    [
        "rank",
        "source",
        "chunk_id",
        "similarity"
    ]
]

,rank,source,chunk_id,similarity
0,1,03_bank_transfers.md,03_bank_transfers.md_0,0.0851
1,2,01_card_payments.md,01_card_payments.md_0,0.0716
2,3,03_bank_transfers.md,03_bank_transfers.md_1,0.0413


Reload all 8 documents

In [41]:
KNOWLEDGE_BASE_DIR = Path(
    "../knowledge_base"
)

documents = []

for file_path in sorted(
    KNOWLEDGE_BASE_DIR.glob("*.md")
):

    text = file_path.read_text(
        encoding="utf-8"
    )

    documents.append({
        "source": file_path.name,
        "content": text
    })

print(
    "Documents loaded:",
    len(documents)
)

Documents loaded: 8


In [42]:
all_chunks = []

for document in documents:

    document_chunks = (
        chunk_markdown_document(
            document["content"],
            document["source"]
        )
    )

    all_chunks.extend(
        document_chunks
    )

print(
    "Total chunks:",
    len(all_chunks)
)

Total chunks: 40


Inspect the expanded knowledge base

In [43]:
chunks_df = pd.DataFrame(
    all_chunks
)

print(
    chunks_df.groupby("source")
    .size()
)

source
01_card_payments.md            5
02_refunds.md                  5
03_bank_transfers.md           5
04_cash_withdrawals.md         5
05_cards_and_pin.md            5
06_account_security.md         5
07_identity_verification.md    5
08_support_escalation.md       5
dtype: int64


Rebuild the embeddings

In [44]:
chunk_texts = [
    chunk["text"]
    for chunk in all_chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True
)

print(
    "Chunk embeddings shape:",
    chunk_embeddings.shape
)

Chunk embeddings shape: (40, 384)


Rebuild ChromaDB

In [45]:
COLLECTION_NAME = (
    "novabank_support_policies"
)

try:
    chroma_client.delete_collection(
        name=COLLECTION_NAME
    )
except Exception:
    pass

collection = (
    chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={
            "hnsw:space": "cosine"
        }
    )
)

In [46]:
chunk_ids = [
    chunk["chunk_id"]
    for chunk in all_chunks
]

chunk_metadata = [
    {
        "source":
            chunk["source"],

        "chunk_id":
            chunk["chunk_id"]
    }
    for chunk in all_chunks
]

In [47]:
collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    embeddings=chunk_embeddings.tolist(),
    metadatas=chunk_metadata
)

print(
    "Chunks stored in ChromaDB:",
    collection.count()
)

Chunks stored in ChromaDB: 40


Test the new topics

In [48]:
test_questions = [
    "The ATM charged my account but did not give me any cash.",
    "My PIN is blocked after I entered it incorrectly.",
    "I lost my card and I can see a transaction I do not recognise.",
    "My identity verification keeps failing.",
    "When should a support ticket be escalated?"
]

for question in test_questions:

    print("\n" + "=" * 90)
    print("QUESTION:")
    print(question)

    results = retrieve_context(
        question,
        top_k=3
    )

    for item in results:

        print(
            f"\nRank {item['rank']} | "
            f"{item['source']} | "
            f"{item['chunk_id']} | "
            f"Similarity: "
            f"{item['similarity']}"
        )


QUESTION:
The ATM charged my account but did not give me any cash.

Rank 1 | 04_cash_withdrawals.md | 04_cash_withdrawals.md_2 | Similarity: 0.5956

Rank 2 | 04_cash_withdrawals.md | 04_cash_withdrawals.md_3 | Similarity: 0.5677

Rank 3 | 04_cash_withdrawals.md | 04_cash_withdrawals.md_4 | Similarity: 0.4998

QUESTION:
My PIN is blocked after I entered it incorrectly.

Rank 1 | 05_cards_and_pin.md | 05_cards_and_pin.md_2 | Similarity: 0.6767

Rank 2 | 05_cards_and_pin.md | 05_cards_and_pin.md_3 | Similarity: 0.4634

Rank 3 | 06_account_security.md | 06_account_security.md_1 | Similarity: 0.4245

QUESTION:
I lost my card and I can see a transaction I do not recognise.

Rank 1 | 06_account_security.md | 06_account_security.md_2 | Similarity: 0.5898

Rank 2 | 01_card_payments.md | 01_card_payments.md_3 | Similarity: 0.5388

Rank 3 | 05_cards_and_pin.md | 05_cards_and_pin.md_4 | Similarity: 0.4855

QUESTION:
My identity verification keeps failing.

Rank 1 | 07_identity_verification.md | 0

Expand the retrieval evaluation

In [49]:
retrieval_tests = [
    {
        "question":
            "I was charged twice for the same transaction.",
        "expected_source":
            "01_card_payments.md"
    },

    {
        "question":
            "The merchant refunded me but the money has not arrived.",
        "expected_source":
            "02_refunds.md"
    },

    {
        "question":
            "My bank transfer has been pending for three days.",
        "expected_source":
            "03_bank_transfers.md"
    },

    {
        "question":
            "The ATM charged me but did not give me any cash.",
        "expected_source":
            "04_cash_withdrawals.md"
    },

    {
        "question":
            "My PIN has been blocked after too many attempts.",
        "expected_source":
            "05_cards_and_pin.md"
    },

    {
        "question":
            "I lost my card and think somebody may be using it.",
        "expected_source":
            "06_account_security.md"
    },

    {
        "question":
            "My identity verification keeps failing.",
        "expected_source":
            "07_identity_verification.md"
    },

    {
        "question":
            "When should a customer support case be escalated?",
        "expected_source":
            "08_support_escalation.md"
    }
]

In [50]:
evaluation_rows = []

for test in retrieval_tests:

    retrieved = retrieve_context(
        test["question"],
        top_k=3
    )

    retrieved_sources = [
        item["source"]
        for item in retrieved
    ]

    evaluation_rows.append({
        "question":
            test["question"],

        "expected_source":
            test["expected_source"],

        "top_1_source":
            retrieved_sources[0],

        "hit_at_1":
            retrieved_sources[0]
            == test["expected_source"],

        "hit_at_3":
            test["expected_source"]
            in retrieved_sources
    })

retrieval_evaluation_df = pd.DataFrame(
    evaluation_rows
)

retrieval_evaluation_df

,question,expected_source,top_1_source,hit_at_1,hit_at_3
0,I was charged twice for the same transaction.,01_card_payments.md,02_refunds.md,False,True
1,The merchant refunded me but the money has not...,02_refunds.md,02_refunds.md,True,True
2,My bank transfer has been pending for three days.,03_bank_transfers.md,03_bank_transfers.md,True,True
3,The ATM charged me but did not give me any cash.,04_cash_withdrawals.md,04_cash_withdrawals.md,True,True
4,My PIN has been blocked after too many attempts.,05_cards_and_pin.md,05_cards_and_pin.md,True,True
5,I lost my card and think somebody may be using...,06_account_security.md,05_cards_and_pin.md,False,True
6,My identity verification keeps failing.,07_identity_verification.md,07_identity_verification.md,True,True
7,When should a customer support case be escalated?,08_support_escalation.md,08_support_escalation.md,True,True


In [51]:
hit_at_1 = (
    retrieval_evaluation_df[
        "hit_at_1"
    ].mean()
)

hit_at_3 = (
    retrieval_evaluation_df[
        "hit_at_3"
    ].mean()
)

print(
    f"Retrieval Hit@1: "
    f"{hit_at_1:.2%}"
)

print(
    f"Retrieval Hit@3: "
    f"{hit_at_3:.2%}"
)

Retrieval Hit@1: 75.00%
Retrieval Hit@3: 100.00%


Saving the retrieval evaluation

In [52]:
from pathlib import Path

RAG_EVAL_DIR = Path("../rag/evaluation")
RAG_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

retrieval_evaluation_df.to_csv(
    RAG_EVAL_DIR / "retrieval_evaluation.csv",
    index=False
)

print("Retrieval evaluation saved.")

Retrieval evaluation saved.


In [53]:
retrieval_evaluation_df[
    retrieval_evaluation_df["hit_at_1"] == False
]

,question,expected_source,top_1_source,hit_at_1,hit_at_3
0,I was charged twice for the same transaction.,01_card_payments.md,02_refunds.md,False,True
5,I lost my card and think somebody may be using...,06_account_security.md,05_cards_and_pin.md,False,True


Load the key safely

In [67]:
import os

from dotenv import load_dotenv
from google import genai

In [68]:
load_dotenv("../.env")

gemini_api_key = os.getenv(
    "GEMINI_API_KEY"
)

if not gemini_api_key:
    raise ValueError(
        "GEMINI_API_KEY was not found in the .env file."
    )

print("Gemini API key loaded successfully!")

Gemini API key loaded successfully!


Create the Gemini client

In [69]:
client = genai.Client(
    api_key=gemini_api_key
)

print("Gemini client created successfully!")

Gemini client created successfully!


Test Gemini before touching RAG

In [70]:
LLM_MODEL = "gemini-3.7-flash"

In [73]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")

gemini_api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(
    api_key=gemini_api_key,
    http_options=types.HttpOptions(
        timeout=30000
    )
)

print("Gemini client recreated with timeout.")

Gemini client recreated with timeout.


In [74]:
model_info = client.models.get(
    model="gemini-3.7-flash"
)

print(model_info.name)

models/gemini-3.7-flash


In [78]:
test_response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply only with: Gemini connection successful",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level="minimal"
        ),
        max_output_tokens=20
    )
)

print(test_response.text)

Gemini connection successful


In [79]:
LLM_MODEL = "gemini-3.5-flash-lite"

print("LLM model:", LLM_MODEL)

LLM model: gemini-3.5-flash-lite


Build the grounded RAG prompt

In [80]:
def build_rag_prompt(
    question,
    context
):

    prompt = f"""
You are SupportOps AI, an internal customer-support assistant for NovaBank.

Your task is to answer the support question using ONLY the NovaBank policy
information provided in the retrieved context.

GROUNDING RULES

1. Use only information contained in the retrieved policy context.
2. Do not invent policies, timelines, fees, requirements, or procedures.
3. If the context does not contain enough information to answer the question,
   clearly say:
   "The available policy context does not provide enough information."
4. Give clear and actionable steps for the support agent.
5. Mention timelines only when they are explicitly stated in the policy.
6. Cite supporting policy evidence using [Source 1], [Source 2], etc.
7. Do not cite a source unless it supports the statement.
8. Do not claim that an investigation, refund, escalation, or other action
   has already been completed.
9. Keep the answer concise and professional.

RETRIEVED NOVABANK POLICY CONTEXT
---------------------------------
{context}

SUPPORT QUESTION
----------------
{question}

GROUNDED SUPPORT RECOMMENDATION
-------------------------------
"""

    return prompt.strip()

Make sure your context builder has source labels

In [81]:
def build_rag_context(
    query,
    top_k=3
):

    retrieved = retrieve_context(
        query,
        top_k=top_k
    )

    context_blocks = []

    for index, item in enumerate(
        retrieved,
        start=1
    ):

        block = (
            f"[Source {index}: "
            f"{item['source']} | "
            f"{item['chunk_id']}]\n"
            f"{item['text']}"
        )

        context_blocks.append(
            block
        )

    context = "\n\n---\n\n".join(
        context_blocks
    )

    return context, retrieved

Create the Gemini generation function

In [82]:
def generate_grounded_answer(
    question,
    context
):

    prompt = build_rag_prompt(
        question,
        context
    )

    response = client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt
    )

    return response.text

Build the complete RAG function

In [83]:
def answer_with_rag(
    question,
    top_k=3
):

    # STEP 1
    # Retrieve relevant NovaBank policy chunks
    context, retrieved = build_rag_context(
        question,
        top_k=top_k
    )

    # STEP 2
    # Send retrieved context + question to Gemini
    answer = generate_grounded_answer(
        question,
        context
    )

    # STEP 3
    # Return both answer and evidence
    return {
        "question": question,
        "answer": answer,
        "sources": retrieved
    }

Test 1: Duplicate card payment

In [84]:
rag_result = answer_with_rag(
    "I was charged twice for the same card transaction. "
    "What should the support agent do?"
)

In [85]:
print(
    rag_result["answer"]
)

Based on NovaBank's policies, here are the actionable steps for the support agent:

1. Instruct the customer to verify whether both transactions have fully completed, as one transaction may temporarily remain pending before disappearing [Source 1].
2. If both transactions are completed and represent the same purchase, create a duplicate-payment investigation [Source 1, Source 2].
3. Collect the required information from the customer: the merchant name, transaction date, transaction amount, the last four digits of the affected card, and evidence that both transactions represent the same purchase [Source 1, Source 2].
4. Inform the customer that duplicate-payment investigations are normally reviewed within five business days [Source 1].


Display answer + retrieval together

In [86]:
print("QUESTION")
print("=" * 80)

print(
    rag_result["question"]
)


print("\nANSWER")
print("=" * 80)

print(
    rag_result["answer"]
)


print("\nRETRIEVED SOURCES")
print("=" * 80)

for source in rag_result["sources"]:

    print(
        f"Rank {source['rank']} | "
        f"{source['source']} | "
        f"{source['chunk_id']} | "
        f"Similarity: "
        f"{source['similarity']}"
    )

QUESTION
I was charged twice for the same card transaction. What should the support agent do?

ANSWER
Based on NovaBank's policies, here are the actionable steps for the support agent:

1. Instruct the customer to verify whether both transactions have fully completed, as one transaction may temporarily remain pending before disappearing [Source 1].
2. If both transactions are completed and represent the same purchase, create a duplicate-payment investigation [Source 1, Source 2].
3. Collect the required information from the customer: the merchant name, transaction date, transaction amount, the last four digits of the affected card, and evidence that both transactions represent the same purchase [Source 1, Source 2].
4. Inform the customer that duplicate-payment investigations are normally reviewed within five business days [Source 1].

RETRIEVED SOURCES
Rank 1 | 01_card_payments.md | 01_card_payments.md_2 | Similarity: 0.6171
Rank 2 | 02_refunds.md | 02_refunds.md_3 | Similarity: 0.607

Test 2: Pending bank transfer

In [87]:
transfer_result = answer_with_rag(
    "A customer's bank transfer has been pending "
    "for three business days. "
    "What should the support agent do?"
)

print(
    transfer_result["answer"]
)

Based on NovaBank policy, since the bank transfer has been pending for more than the allowed two business days, the support agent should escalate the case to the Transfers team [Source 1].


In [88]:
for source in transfer_result["sources"]:

    print(
        source["rank"],
        source["source"],
        source["chunk_id"],
        source["similarity"]
    )

1 03_bank_transfers.md 03_bank_transfers.md_1 0.6561
2 03_bank_transfers.md 03_bank_transfers.md_3 0.5096
3 03_bank_transfers.md 03_bank_transfers.md_2 0.4904


Test 3: ATM cash problem

In [89]:
atm_result = answer_with_rag(
    "The ATM charged the customer's account "
    "but did not dispense any cash. "
    "What should the agent do?"
)

print(
    atm_result["answer"]
)

Based on the NovaBank policy, if the customer did not receive cash from the ATM even though the transaction is shown as completed, the support agent should follow these steps:

1. Instruct the customer not to immediately repeat the same withdrawal [Source 2].
2. Record the following details [Source 2]:
   - ATM location [Source 2]
   - Withdrawal date [Source 2]
   - Withdrawal amount [Source 2]
   - Last four digits of the card [Source 2]
3. Submit the case for a cash-withdrawal investigation [Source 2]. 

*Note: Cash-withdrawal investigations are normally reviewed within seven business days [Source 2].*


Test hallucination resistance

In [90]:
unsupported_result = answer_with_rag(
    "What interest rate does NovaBank pay "
    "on savings accounts?"
)

print(
    unsupported_result["answer"]
)

The available policy context does not provide enough information.


Completely unrelated question

In [91]:
weather_result = answer_with_rag(
    "What is the weather going to be tomorrow?"
)

print(
    weather_result["answer"]
)

The available policy context does not provide enough information.


Inspect the unsupported retrieval

In [92]:
for source in weather_result["sources"]:

    print(
        f"Rank {source['rank']} | "
        f"{source['source']} | "
        f"Similarity: "
        f"{source['similarity']}"
    )

Rank 1 | 01_card_payments.md | Similarity: 0.0955
Rank 2 | 05_cards_and_pin.md | Similarity: 0.0948
Rank 3 | 03_bank_transfers.md | Similarity: 0.0885


Save our retrieval evaluation

In [93]:
from pathlib import Path

RAG_EVAL_DIR = Path(
    "../rag/evaluation"
)

RAG_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [94]:
retrieval_evaluation_df.to_csv(
    RAG_EVAL_DIR
    / "retrieval_evaluation.csv",
    index=False
)

print(
    "Retrieval evaluation saved!"
)

Retrieval evaluation saved!


Create in-domain questions

In [95]:
in_domain_questions = [
    "I was charged twice for the same card payment.",
    "My transfer has been pending for three days.",
    "The ATM charged me but gave me no cash.",
    "My PIN is blocked.",
    "I lost my card and someone may have used it.",
    "My identity verification keeps failing.",
    "The merchant refunded me but the money has not arrived.",
    "When should this support case be escalated?"
]

Create out-of-domain questions

In [96]:
out_of_domain_questions = [
    "What is the weather tomorrow?",
    "Who won the football match yesterday?",
    "What is the capital of France?",
    "How do I cook pasta?",
    "What interest rate does NovaBank offer on savings accounts?",
    "Can you recommend a good laptop?",
    "What is the stock price of Apple?",
    "Write me a birthday message."
]

Measure top retrieval similarity

In [97]:
def get_top_similarity(question):
    
    results = retrieve_context(
        question,
        top_k=1
    )
    
    if not results:
        return 0.0
    
    return results[0]["similarity"]

In [98]:
print(
    get_top_similarity(
        "My PIN is blocked."
    )
)

0.6819


Compare in-domain vs out-of-domain

In [99]:
confidence_rows = []

for question in in_domain_questions:
    
    confidence_rows.append({
        "question": question,
        "category": "in_domain",
        "top_similarity":
            get_top_similarity(question)
    })


for question in out_of_domain_questions:
    
    confidence_rows.append({
        "question": question,
        "category": "out_of_domain",
        "top_similarity":
            get_top_similarity(question)
    })


confidence_df = pd.DataFrame(
    confidence_rows
)

confidence_df

,question,category,top_similarity
0,I was charged twice for the same card payment.,in_domain,0.6341
1,My transfer has been pending for three days.,in_domain,0.6564
2,The ATM charged me but gave me no cash.,in_domain,0.5581
3,My PIN is blocked.,in_domain,0.6819
4,I lost my card and someone may have used it.,in_domain,0.4980
5,My identity verification keeps failing.,in_domain,0.6346
6,The merchant refunded me but the money has not...,in_domain,0.5608
7,When should this support case be escalated?,in_domain,0.7408
8,What is the weather tomorrow?,out_of_domain,0.0893
9,Who won the football match yesterday?,out_of_domain,0.1229


Summarize the distributions

In [100]:
confidence_summary = (
    confidence_df
    .groupby("category")[
        "top_similarity"
    ]
    .agg([
        "min",
        "mean",
        "max"
    ])
)

confidence_summary

,min,mean,max
category,,,
in_domain,0.4980,0.620587,0.7408
out_of_domain,0.0291,0.167550,0.6319


Inspect the hardest cases

In [101]:
print("LOWEST IN-DOMAIN SCORES")
print("=" * 70)

display(
    confidence_df[
        confidence_df["category"]
        == "in_domain"
    ]
    .sort_values(
        "top_similarity"
    )
    .head(5)
)


print("\nHIGHEST OUT-OF-DOMAIN SCORES")
print("=" * 70)

display(
    confidence_df[
        confidence_df["category"]
        == "out_of_domain"
    ]
    .sort_values(
        "top_similarity",
        ascending=False
    )
    .head(5)
)

LOWEST IN-DOMAIN SCORES


,question,category,top_similarity
4,I lost my card and someone may have used it.,in_domain,0.4980
2,The ATM charged me but gave me no cash.,in_domain,0.5581
6,The merchant refunded me but the money has not...,in_domain,0.5608
0,I was charged twice for the same card payment.,in_domain,0.6341
5,My identity verification keeps failing.,in_domain,0.6346



HIGHEST OUT-OF-DOMAIN SCORES


,question,category,top_similarity
12,What interest rate does NovaBank offer on savi...,out_of_domain,0.6319
14,What is the stock price of Apple?,out_of_domain,0.1925
9,Who won the football match yesterday?,out_of_domain,0.1229
15,Write me a birthday message.,out_of_domain,0.1228
10,What is the capital of France?,out_of_domain,0.0957


Save this experiment

In [102]:
confidence_df.to_csv(
    "../rag/evaluation/"
    "retrieval_confidence_analysis.csv",
    index=False
)

print(
    "Confidence analysis saved!"
)

Confidence analysis saved!


Update the question categories

In [103]:
supported_questions = [
    "I was charged twice for the same card payment.",
    "My transfer has been pending for three days.",
    "The ATM charged me but gave me no cash.",
    "My PIN is blocked.",
    "I lost my card and someone may have used it.",
    "My identity verification keeps failing.",
    "The merchant refunded me but the money has not arrived.",
    "When should this support case be escalated?"
]

unsupported_in_domain_questions = [
    "What interest rate does NovaBank offer on savings accounts?"
]

out_of_domain_questions = [
    "What is the weather tomorrow?",
    "Who won the football match yesterday?",
    "What is the capital of France?",
    "How do I cook pasta?",
    "Can you recommend a good laptop?",
    "What is the stock price of Apple?",
    "Write me a birthday message."
]

Add threshold guardrail

In [104]:
RETRIEVAL_THRESHOLD = 0.45

In [105]:
def format_retrieved_context(
    retrieved
):

    context_blocks = []

    for index, item in enumerate(
        retrieved,
        start=1
    ):

        block = (
            f"[Source {index}: "
            f"{item['source']} | "
            f"{item['chunk_id']}]\n"
            f"{item['text']}"
        )

        context_blocks.append(block)

    return "\n\n---\n\n".join(
        context_blocks
    )

In [106]:
def answer_with_rag(
    question,
    top_k=3,
    threshold=RETRIEVAL_THRESHOLD
):

    # STEP 1
    # Retrieve relevant policy chunks
    retrieved = retrieve_context(
        question,
        top_k=top_k
    )

    # Handle empty retrieval
    if not retrieved:

        return {
            "question": question,
            "answer": (
                "The available policy context does not "
                "provide enough information."
            ),
            "sources": [],
            "top_similarity": 0.0,
            "status": "no_retrieval"
        }

    # STEP 2
    # Inspect strongest retrieval result
    top_similarity = (
        retrieved[0]["similarity"]
    )

    # STEP 3
    # Reject clearly unrelated questions
    if top_similarity < threshold:

        return {
            "question": question,
            "answer": (
                "The available policy context does not "
                "provide enough information."
            ),
            "sources": retrieved,
            "top_similarity":
                top_similarity,
            "status":
                "low_retrieval_confidence"
        }

    # STEP 4
    # Build context from retrieved evidence
    context = format_retrieved_context(
        retrieved
    )

    # STEP 5
    # Generate grounded response with Gemini
    answer = generate_grounded_answer(
        question,
        context
    )

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved,
        "top_similarity":
            top_similarity,
        "status":
            "generated"
    }

Test the guardrail

In [107]:
result = answer_with_rag(
    "My PIN is blocked."
)

print("Status:", result["status"])
print(
    "Similarity:",
    result["top_similarity"]
)
print(
    "Answer:",
    result["answer"]
)

Status: generated
Similarity: 0.6819
Answer: Based on the NovaBank policy, here are the actionable steps for the support agent:

1. **Verify Identity:** Verify the customer's identity before providing available PIN recovery instructions [Source 1].
2. **Advise the Customer:** Instruct the customer not to continue entering possible PIN combinations [Source 1].
3. **Attempt Recovery:** Guide the customer through the supported PIN recovery process. 
4. **Escalate if Necessary:** If the PIN cannot be restored using the supported recovery process, escalate the case to the Cards team [Source 1].


Weather question

In [108]:
result = answer_with_rag(
    "What is the weather tomorrow?"
)

print("Status:", result["status"])
print(
    "Similarity:",
    result["top_similarity"]
)
print(
    "Answer:",
    result["answer"]
)

Status: low_retrieval_confidence
Similarity: 0.0893
Answer: The available policy context does not provide enough information.


Savings-interest question

In [109]:
result = answer_with_rag(
    "What interest rate does NovaBank "
    "offer on savings accounts?"
)

print("Status:", result["status"])
print(
    "Similarity:",
    result["top_similarity"]
)
print(
    "Answer:",
    result["answer"]
)

Status: generated
Similarity: 0.6319
Answer: The available policy context does not provide enough information.


Larger RAG evaluation

In [110]:
rag_evaluation_questions = [
    # --------------------------------------------------
    # CARD PAYMENTS
    # --------------------------------------------------
    {
        "question":
            "Why was my card payment declined even though I have enough money?",
        "expected_source":
            "01_card_payments.md",
        "category":
            "supported"
    },
    {
        "question":
            "I can see the same card purchase twice on my account.",
        "expected_source":
            "01_card_payments.md",
        "category":
            "supported"
    },
    {
        "question":
            "There is a card transaction I do not recognise.",
        "expected_source":
            "01_card_payments.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # REFUNDS
    # --------------------------------------------------
    {
        "question":
            "The merchant says they refunded me but I still cannot see it.",
        "expected_source":
            "02_refunds.md",
        "category":
            "supported"
    },
    {
        "question":
            "How long should I wait for a merchant refund to appear?",
        "expected_source":
            "02_refunds.md",
        "category":
            "supported"
    },
    {
        "question":
            "The merchant processed my refund more than ten business days ago.",
        "expected_source":
            "02_refunds.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # TRANSFERS
    # --------------------------------------------------
    {
        "question":
            "My transfer has been pending for three business days.",
        "expected_source":
            "03_bank_transfers.md",
        "category":
            "supported"
    },
    {
        "question":
            "My bank transfer keeps failing.",
        "expected_source":
            "03_bank_transfers.md",
        "category":
            "supported"
    },
    {
        "question":
            "The transfer says completed but the recipient has not received it.",
        "expected_source":
            "03_bank_transfers.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # CASH WITHDRAWALS
    # --------------------------------------------------
    {
        "question":
            "The ATM charged my account but did not give me cash.",
        "expected_source":
            "04_cash_withdrawals.md",
        "category":
            "supported"
    },
    {
        "question":
            "The ATM gave me less cash than the amount shown on my account.",
        "expected_source":
            "04_cash_withdrawals.md",
        "category":
            "supported"
    },
    {
        "question":
            "My cash withdrawal keeps getting declined.",
        "expected_source":
            "04_cash_withdrawals.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # CARDS AND PIN
    # --------------------------------------------------
    {
        "question":
            "My PIN is blocked after several incorrect attempts.",
        "expected_source":
            "05_cards_and_pin.md",
        "category":
            "supported"
    },
    {
        "question":
            "My new card will not activate.",
        "expected_source":
            "05_cards_and_pin.md",
        "category":
            "supported"
    },
    {
        "question":
            "My physical card is damaged and no longer works.",
        "expected_source":
            "05_cards_and_pin.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # SECURITY
    # --------------------------------------------------
    {
        "question":
            "I lost my card and I think someone has used it.",
        "expected_source":
            "06_account_security.md",
        "category":
            "supported"
    },
    {
        "question":
            "I think someone has gained access to my account.",
        "expected_source":
            "06_account_security.md",
        "category":
            "supported"
    },
    {
        "question":
            "I lost the phone I use to access NovaBank.",
        "expected_source":
            "06_account_security.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # IDENTITY VERIFICATION
    # --------------------------------------------------
    {
        "question":
            "My identity verification keeps failing.",
        "expected_source":
            "07_identity_verification.md",
        "category":
            "supported"
    },
    {
        "question":
            "My account name does not match my identity document.",
        "expected_source":
            "07_identity_verification.md",
        "category":
            "supported"
    },
    {
        "question":
            "What happens when automated verification repeatedly fails?",
        "expected_source":
            "07_identity_verification.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # ESCALATION
    # --------------------------------------------------
    {
        "question":
            "When should a support case be escalated?",
        "expected_source":
            "08_support_escalation.md",
        "category":
            "supported"
    },
    {
        "question":
            "What information should an escalation contain?",
        "expected_source":
            "08_support_escalation.md",
        "category":
            "supported"
    },
    {
        "question":
            "How should suspected account takeover be escalated?",
        "expected_source":
            "08_support_escalation.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # UNSUPPORTED BUT BANKING RELATED
    # --------------------------------------------------
    {
        "question":
            "What interest rate does NovaBank pay on savings accounts?",
        "expected_source":
            None,
        "category":
            "unsupported_in_domain"
    },
    {
        "question":
            "Does NovaBank offer mortgages?",
        "expected_source":
            None,
        "category":
            "unsupported_in_domain"
    },
    {
        "question":
            "What is NovaBank's overdraft interest rate?",
        "expected_source":
            None,
        "category":
            "unsupported_in_domain"
    },
    {
        "question":
            "Can I open a business banking account?",
        "expected_source":
            None,
        "category":
            "unsupported_in_domain"
    },

    # --------------------------------------------------
    # TRUE OUT OF DOMAIN
    # --------------------------------------------------
    {
        "question":
            "What is the weather tomorrow?",
        "expected_source":
            None,
        "category":
            "out_of_domain"
    },
    {
        "question":
            "Who won yesterday's football match?",
        "expected_source":
            None,
        "category":
            "out_of_domain"
    },
    {
        "question":
            "Can you recommend a laptop?",
        "expected_source":
            None,
        "category":
            "out_of_domain"
    },
    {
        "question":
            "Write a birthday message for my friend.",
        "expected_source":
            None,
        "category":
            "out_of_domain"
    }
]

print(
    "Evaluation questions:",
    len(rag_evaluation_questions)
)

Evaluation questions: 32


Evaluate retrieval without Gemini

In [111]:
retrieval_eval_rows = []

for item in rag_evaluation_questions:

    retrieved = retrieve_context(
        item["question"],
        top_k=3
    )

    top_similarity = (
        retrieved[0]["similarity"]
        if retrieved
        else 0.0
    )

    retrieved_sources = [
        result["source"]
        for result in retrieved
    ]

    if item["category"] == "supported":

        hit_at_1 = (
            retrieved_sources[0]
            == item["expected_source"]
        )

        hit_at_3 = (
            item["expected_source"]
            in retrieved_sources
        )

    else:
        hit_at_1 = None
        hit_at_3 = None

    retrieval_eval_rows.append({
        "question":
            item["question"],

        "category":
            item["category"],

        "expected_source":
            item["expected_source"],

        "top_source":
            retrieved_sources[0]
            if retrieved_sources
            else None,

        "top_similarity":
            top_similarity,

        "hit_at_1":
            hit_at_1,

        "hit_at_3":
            hit_at_3,

        "passes_threshold":
            top_similarity
            >= RETRIEVAL_THRESHOLD
    })


rag_eval_df = pd.DataFrame(
    retrieval_eval_rows
)

rag_eval_df

,question,category,expected_source,top_source,top_similarity,hit_at_1,hit_at_3,passes_threshold
0,Why was my card payment declined even though I...,supported,01_card_payments.md,01_card_payments.md,0.7087,True,True,True
1,I can see the same card purchase twice on my a...,supported,01_card_payments.md,01_card_payments.md,0.5972,True,True,True
2,There is a card transaction I do not recognise.,supported,01_card_payments.md,01_card_payments.md,0.6073,True,True,True
3,The merchant says they refunded me but I still...,supported,02_refunds.md,02_refunds.md,0.4554,True,True,True
4,How long should I wait for a merchant refund t...,supported,02_refunds.md,02_refunds.md,0.7916,True,True,True
5,The merchant processed my refund more than ten...,supported,02_refunds.md,02_refunds.md,0.6719,True,True,True
6,My transfer has been pending for three busines...,supported,03_bank_transfers.md,03_bank_transfers.md,0.6836,True,True,True
7,My bank transfer keeps failing.,supported,03_bank_transfers.md,03_bank_transfers.md,0.6117,True,True,True
8,The transfer says completed but the recipient ...,supported,03_bank_transfers.md,03_bank_transfers.md,0.7164,True,True,True
9,The ATM charged my account but did not give me...,supported,04_cash_withdrawals.md,04_cash_withdrawals.md,0.6064,True,True,True


Calculate retrieval metrics

In [112]:
supported_eval = rag_eval_df[
    rag_eval_df["category"]
    == "supported"
].copy()

In [113]:
retrieval_hit_at_1 = (
    supported_eval[
        "hit_at_1"
    ].mean()
)

retrieval_hit_at_3 = (
    supported_eval[
        "hit_at_3"
    ].mean()
)

print(
    f"Retrieval Hit@1: "
    f"{retrieval_hit_at_1:.2%}"
)

print(
    f"Retrieval Hit@3: "
    f"{retrieval_hit_at_3:.2%}"
)

Retrieval Hit@1: 100.00%
Retrieval Hit@3: 100.00%


Test whether threshold rejects valid questions

In [114]:
supported_pass_rate = (
    supported_eval[
        "passes_threshold"
    ].mean()
)

print(
    f"Supported-query pass rate: "
    f"{supported_pass_rate:.2%}"
)

Supported-query pass rate: 100.00%


Evaluate true out-of-domain rejection

In [115]:
ood_eval = rag_eval_df[
    rag_eval_df["category"]
    == "out_of_domain"
].copy()

ood_rejection_rate = (
    ~ood_eval[
        "passes_threshold"
    ]
).mean()

print(
    f"Out-of-domain rejection rate: "
    f"{ood_rejection_rate:.2%}"
)

Out-of-domain rejection rate: 100.00%


Inspect unsupported banking queries

In [116]:
unsupported_eval = rag_eval_df[
    rag_eval_df["category"]
    == "unsupported_in_domain"
].copy()

unsupported_eval[
    [
        "question",
        "top_source",
        "top_similarity",
        "passes_threshold"
    ]
]

,question,top_source,top_similarity,passes_threshold
24,What interest rate does NovaBank pay on saving...,03_bank_transfers.md,0.6278,True
25,Does NovaBank offer mortgages?,01_card_payments.md,0.6099,True
26,What is NovaBank's overdraft interest rate?,01_card_payments.md,0.6097,True
27,Can I open a business banking account?,06_account_security.md,0.3603,False


Save larger retrieval evaluation

In [117]:
rag_eval_df.to_csv(
    "../rag/evaluation/"
    "rag_retrieval_evaluation_32_queries.csv",
    index=False
)

print(
    "32-query RAG retrieval evaluation saved!"
)

32-query RAG retrieval evaluation saved!


Citation validation

In [118]:
import re


def has_source_citation(answer):

    pattern = r"\[Source\s+\d+\]"

    return bool(
        re.search(
            pattern,
            answer
        )
    )

In [119]:
print(
    has_source_citation(
        "Escalate the issue [Source 1]."
    )
)

print(
    has_source_citation(
        "Escalate the issue."
    )
)

True
False


Check whether citations are valid

In [120]:
def extract_source_numbers(answer):

    matches = re.findall(
        r"\[Source\s+(\d+)\]",
        answer
    )

    return [
        int(number)
        for number in matches
    ]

In [121]:
def citations_are_valid(
    answer,
    number_of_sources
):

    citation_numbers = (
        extract_source_numbers(
            answer
        )
    )

    if not citation_numbers:
        return False

    return all(
        1 <= citation
        <= number_of_sources

        for citation
        in citation_numbers
    )

In [122]:
print(
    citations_are_valid(
        "Follow the policy [Source 1].",
        3
    )
)

True


In [123]:
print(
    citations_are_valid(
        "Follow the policy [Source 7].",
        3
    )
)

False


Small generation-quality evaluation

In [124]:
generation_test_questions = [
    # Supported
    "I was charged twice for the same card payment.",
    "My transfer has been pending for three business days.",
    "The ATM charged my account but did not give me cash.",
    "My PIN is blocked.",
    "I lost my card and think someone may have used it.",
    "My identity verification keeps failing.",
    "What information should an escalation contain?",

    # Unsupported in-domain
    "What interest rate does NovaBank pay on savings accounts?",
    "Does NovaBank offer mortgages?",

    # Out of domain
    "What is the weather tomorrow?",
    "Can you recommend a laptop?",
    "Write me a birthday message."
]

In [125]:
generation_eval_rows = []

for question in generation_test_questions:

    result = answer_with_rag(
        question
    )

    generation_eval_rows.append({
        "question":
            question,

        "status":
            result["status"],

        "top_similarity":
            result["top_similarity"],

        "answer":
            result["answer"],

        "has_citation":
            has_source_citation(
                result["answer"]
            ),

        "valid_citations":
            (
                citations_are_valid(
                    result["answer"],
                    len(result["sources"])
                )
                if result["status"]
                == "generated"
                else True
            )
    })


generation_eval_df = pd.DataFrame(
    generation_eval_rows
)

generation_eval_df

,question,status,top_similarity,answer,has_citation,valid_citations
0,I was charged twice for the same card payment.,generated,0.6341,**Grounded Support Recommendation:**\n\nTo ass...,True,True
1,My transfer has been pending for three busines...,generated,0.6836,**Grounded Support Recommendation:**\n\n1. Inf...,True,True
2,The ATM charged my account but did not give me...,generated,0.6064,To assist the customer who did not receive cas...,True,True
3,My PIN is blocked.,generated,0.6819,"To assist the customer with a blocked PIN, ple...",True,True
4,I lost my card and think someone may have used...,generated,0.4974,"Based on NovaBank policy, here are the actiona...",True,True
5,My identity verification keeps failing.,generated,0.6346,To assist the customer with their repeated ide...,True,True
6,What information should an escalation contain?,generated,0.7363,"Based on the provided NovaBank policy context,...",True,True
7,What interest rate does NovaBank pay on saving...,generated,0.6278,The available policy context does not provide ...,False,False
8,Does NovaBank offer mortgages?,generated,0.6099,The available policy context does not provide ...,False,False
9,What is the weather tomorrow?,low_retrieval_confidence,0.0893,The available policy context does not provide ...,False,True


Save generation evaluation

In [126]:
generation_eval_df.to_csv(
    "../rag/evaluation/"
    "rag_generation_evaluation.csv",
    index=False
)

print(
    "Generation evaluation saved!"
)

Generation evaluation saved!


Define the fallback message once

In [127]:
FALLBACK_MESSAGE = (
    "The available policy context does not "
    "provide enough information."
)

In [128]:
def is_insufficient_context_answer(answer):
    
    return FALLBACK_MESSAGE.lower() in answer.lower()

In [129]:
print(
    is_insufficient_context_answer(
        "The available policy context does not provide enough information."
    )
)

True


Improve answer_with_rag()

In [130]:
def answer_with_rag(
    question,
    top_k=3,
    threshold=RETRIEVAL_THRESHOLD
):

    # --------------------------------------------------
    # STEP 1: Retrieve relevant policy chunks
    # --------------------------------------------------
    retrieved = retrieve_context(
        question,
        top_k=top_k
    )

    if not retrieved:
        return {
            "question": question,
            "answer": FALLBACK_MESSAGE,
            "sources": [],
            "top_similarity": 0.0,
            "status": "no_retrieval"
        }

    # --------------------------------------------------
    # STEP 2: Inspect strongest retrieval result
    # --------------------------------------------------
    top_similarity = retrieved[0]["similarity"]

    # --------------------------------------------------
    # STEP 3: Reject clearly unrelated questions
    # --------------------------------------------------
    if top_similarity < threshold:
        return {
            "question": question,
            "answer": FALLBACK_MESSAGE,
            "sources": retrieved,
            "top_similarity": top_similarity,
            "status": "low_retrieval_confidence"
        }

    # --------------------------------------------------
    # STEP 4: Build retrieved context
    # --------------------------------------------------
    context = format_retrieved_context(
        retrieved
    )

    # --------------------------------------------------
    # STEP 5: Generate grounded response
    # --------------------------------------------------
    answer = generate_grounded_answer(
        question,
        context
    )

    # --------------------------------------------------
    # STEP 6: Determine whether the knowledge base
    # actually supported an answer
    # --------------------------------------------------
    if is_insufficient_context_answer(answer):
        status = "insufficient_context"
    else:
        status = "answered"

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved,
        "top_similarity": top_similarity,
        "status": status
    }

Fix citation evaluation

In [131]:
def evaluate_citation_behavior(
    result
):

    status = result["status"]
    answer = result["answer"]

    has_citation = (
        has_source_citation(answer)
    )

    if status == "answered":

        valid_citations = (
            citations_are_valid(
                answer,
                len(result["sources"])
            )
        )

        citation_behavior_correct = (
            has_citation
            and valid_citations
        )

    else:

        # Refusals should not be forced
        # to invent policy citations.
        valid_citations = None

        citation_behavior_correct = (
            not has_citation
        )

    return {
        "has_citation":
            has_citation,

        "valid_citations":
            valid_citations,

        "citation_behavior_correct":
            citation_behavior_correct
    }

Rerun generation evaluation

In [132]:
generation_test_questions = [
    # Supported
    "I was charged twice for the same card payment.",
    "My transfer has been pending for three business days.",
    "The ATM charged my account but did not give me cash.",
    "My PIN is blocked.",
    "I lost my card and think someone may have used it.",
    "My identity verification keeps failing.",
    "What information should an escalation contain?",

    # Unsupported but banking-related
    "What interest rate does NovaBank pay on savings accounts?",
    "Does NovaBank offer mortgages?",
    "What is NovaBank's overdraft interest rate?",
    "Can I open a business banking account?",

    # Out of domain
    "What is the weather tomorrow?",
    "Can you recommend a laptop?",
    "Write me a birthday message."
]

In [133]:
generation_eval_rows = []

for question in generation_test_questions:

    result = answer_with_rag(
        question
    )

    citation_metrics = (
        evaluate_citation_behavior(
            result
        )
    )

    generation_eval_rows.append({
        "question":
            question,

        "status":
            result["status"],

        "top_similarity":
            result["top_similarity"],

        "answer":
            result["answer"],

        **citation_metrics
    })


generation_eval_df = pd.DataFrame(
    generation_eval_rows
)

generation_eval_df

,question,status,top_similarity,answer,has_citation,valid_citations,citation_behavior_correct
0,I was charged twice for the same card payment.,answered,0.6341,To assist the customer who was charged twice f...,True,True,True
1,My transfer has been pending for three busines...,answered,0.6836,**Grounded Support Recommendation:**\n\n1. Inf...,True,True,True
2,The ATM charged my account but did not give me...,answered,0.6064,**Grounded Support Recommendation:**\n\n1. Adv...,True,True,True
3,My PIN is blocked.,answered,0.6819,"To assist the customer with a blocked PIN, ple...",True,True,True
4,I lost my card and think someone may have used...,answered,0.4974,To assist the customer who has lost their card...,True,True,True
5,My identity verification keeps failing.,answered,0.6346,To assist the customer with their repeated ide...,True,True,True
6,What information should an escalation contain?,answered,0.7363,"Based on NovaBank's support policy, an escalat...",True,True,True
7,What interest rate does NovaBank pay on saving...,insufficient_context,0.6278,The available policy context does not provide ...,False,None,True
8,Does NovaBank offer mortgages?,insufficient_context,0.6099,The available policy context does not provide ...,False,None,True
9,What is NovaBank's overdraft interest rate?,insufficient_context,0.6097,The available policy context does not provide ...,False,None,True


Calculate final generation metrics

In [134]:
answered_df = generation_eval_df[
    generation_eval_df["status"]
    == "answered"
]

In [135]:
citation_compliance_rate = (
    answered_df[
        "citation_behavior_correct"
    ].mean()
)

print(
    f"Citation compliance: "
    f"{citation_compliance_rate:.2%}"
)

Citation compliance: 100.00%


In [136]:
safe_behavior_rate = (
    generation_eval_df[
        "citation_behavior_correct"
    ].mean()
)

print(
    f"Overall citation/refusal behavior: "
    f"{safe_behavior_rate:.2%}"
)

Overall citation/refusal behavior: 100.00%


Save final evaluation

In [137]:
generation_eval_df.to_csv(
    "../rag/evaluation/rag_generation_evaluation.csv",
    index=False
)

print("Final generation evaluation saved!")

Final generation evaluation saved!
